In [22]:
import pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy
import json

In [2]:
annot_test = pd.read_csv("/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/annotations/annotations_test.csv")

In [ ]:
annot_train = pd.read_csv("/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/annotations/annotations_train.csv")
annot_train.head()

In [ ]:
annot_train[annot_train.image_id == '000434271f63a053c4128a0ba6352c7f']

In [ ]:
annot_train

In [ ]:
annot_train[annot_train.image_id == "0005e8e3701dfb1dd93d53e2ff537b6e"]

In [ ]:
annot_train[annot_train.class_name == "Pleural thickening"]


In [ ]:
annot_test.head()

In [ ]:
annot_test.shape

In [ ]:
annot_test.image_id.value_counts()

In [3]:
annot_test[annot_test.class_name == "Edema"]

,image_id,class_name,x_min,y_min,x_max,y_max


In [ ]:
annot_test[annot_test.image_id == "ab9dedb9ff4cd9e80dca74505b599105"].class_name.unique()

In [4]:
img_lbl_test = pd.read_csv("/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/annotations/image_labels_test.csv")

In [10]:
img_lbl_test.head().T

,0,1,2,3,4
image_id,e0dc2e79105ad93532484e956ef8a71a,0aed23e64ebdea798486056b4f174424,aa15cfcfca7605465ca0513902738b95,665c4a6d2693dc0286d65ab479c9b169,42da2c134b53cb5594774d3d29faac59
Aortic enlargement,0,0,0,0,1
Atelectasis,1,0,0,0,0
Calcification,1,0,0,0,1
Cardiomegaly,1,0,0,0,1
Clavicle fracture,0,0,0,0,0
Consolidation,0,1,0,0,0
Edema,0,0,0,0,0
Emphysema,0,0,0,0,0
Enlarged PA,0,0,0,0,0


In [11]:
img_lbl_test.head()

,image_id,Aortic enlargement,Atelectasis,Calcification,Cardiomegaly,Clavicle fracture,Consolidation,Edema,Emphysema,Enlarged PA,...,Pneumothorax,Pulmonary fibrosis,Rib fracture,Other lesion,COPD,Lung tumor,Pneumonia,Tuberculosis,Other disease,No finding
0,e0dc2e79105ad93532484e956ef8a71a,0,1,1,1,0,0,0,0,0,...,1,0,0,0,0,0,1,0,1,0
1,0aed23e64ebdea798486056b4f174424,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,1,0,0,0
2,aa15cfcfca7605465ca0513902738b95,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,1,0,0
3,665c4a6d2693dc0286d65ab479c9b169,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,42da2c134b53cb5594774d3d29faac59,1,0,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


In [13]:
img_lbl_test.Edema.value_counts()

Edema
0    3000
Name: count, dtype: int64

In [15]:
img_lbl_test.columns[1:]

Index(['Aortic enlargement', 'Atelectasis', 'Calcification', 'Cardiomegaly',
       'Clavicle fracture', 'Consolidation', 'Edema', 'Emphysema',
       'Enlarged PA', 'ILD', 'Infiltration', 'Lung Opacity', 'Lung cavity',
       'Lung cyst', 'Mediastinal shift', 'Nodule/Mass', 'Pleural effusion',
       'Pleural thickening', 'Pneumothorax', 'Pulmonary fibrosis',
       'Rib fracture', 'Other lesion', 'COPD', 'Lung tumor', 'Pneumonia',
       'Tuberculosis', 'Other disease', 'No finding'],
      dtype='object')

In [21]:
set(img_lbl_test.columns[1:]) - {'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion',
 'Lung Opacity', 'No Finding', 'Pleural Effusion', 'Pleural Other', 'Pneumonia', 'Pneumothorax', 'Support Devices'}

{'Aortic enlargement',
 'COPD',
 'Calcification',
 'Clavicle fracture',
 'Emphysema',
 'Enlarged PA',
 'ILD',
 'Infiltration',
 'Lung cavity',
 'Lung cyst',
 'Lung tumor',
 'Mediastinal shift',
 'No finding',
 'Nodule/Mass',
 'Other disease',
 'Other lesion',
 'Pleural effusion',
 'Pleural thickening',
 'Pulmonary fibrosis',
 'Rib fracture',
 'Tuberculosis'}

In [ ]:
images_root = "/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr"
train_meta = pd.read_csv(Path(images_root) / "train_meta.csv")

In [ ]:
train_meta.head()

In [ ]:
x, y = train_meta.loc[train_meta['image_id'] == '4d390e07733ba06e5ff07412f09c0a92', 'dim0'].values, train_meta.loc[train_meta['image_id'] == '4d390e07733ba06e5ff07412f09c0a92', 'dim1'].values

In [ ]:
x, y

In [2]:
def load_meta(meta_csv):
    """
    Load VinDr meta CSV with columns: image_id, dim0, dim1
    Returns dict: image_id -> (orig_height, orig_width)
    """
    df = pd.read_csv(meta_csv)
    meta = {}
    for _, r in df.iterrows():
        image_id = r["image_id"]
        # dim0 = height, dim1 = width (typical numpy convention)
        orig_h = int(r["dim0"])
        orig_w = int(r["dim1"])
        meta[image_id] = (orig_h, orig_w)
    return meta

In [3]:
def visualize_vindr_boxes(
    image_id: str,
    images_root: str,
    annotations_csv: str,
    meta_csv: str,
    save_path: str = None,
):
    """
    Visualize a 256x256 VinDr PNG with its bounding boxes,
    scaling the original annotations using train_meta.csv.
    """
    # Load annotations
    df = pd.read_csv(annotations_csv)
    rows = df[df["image_id"] == image_id]
    if rows.empty:
        raise ValueError(f"No annotations found for image_id={image_id} in {annotations_csv}")

    # Load meta for original size
    meta = load_meta(meta_csv)
    if image_id not in meta:
        raise KeyError(f"image_id={image_id} not found in meta CSV {meta_csv}")
    orig_h, orig_w = meta[image_id]

    # Load 256x256 PNG
    img_path = Path(images_root) / f"{image_id}.png"
    if not img_path.exists():
        raise FileNotFoundError(f"PNG not found: {img_path}")
    img = Image.open(img_path).convert("RGB")
    w_png, h_png = img.size

    assert w_png == h_png == 256, f"Expected 256x256 PNG, got {w_png}x{h_png}"

    # Scaling factors from original -> 256x256
    scale_x = w_png / float(orig_w)
    scale_y = h_png / float(orig_h)

    fig, ax = plt.subplots(1, figsize=(6, 6))
    ax.imshow(img, cmap="gray")
    ax.axis("off")

    for _, r in rows.iterrows():
        x_min_orig = r["x_min"]
        y_min_orig = r["y_min"]
        x_max_orig = r["x_max"]
        y_max_orig = r["y_max"]
        label = r.get("class_name", "")

        # Scale to PNG coordinates
        x_min = x_min_orig * scale_x
        y_min = y_min_orig * scale_y
        x_max = x_max_orig * scale_x
        y_max = y_max_orig * scale_y

        width = x_max - x_min
        height = y_max - y_min

        rect = patches.Rectangle(
            (x_min, y_min),
            width,
            height,
            linewidth=2,
            edgecolor="red",
            facecolor="none",
        )
        ax.add_patch(rect)
        if label:
            ax.text(
                x_min,
                max(y_min - 2, 0),
                label,
                fontsize=8,
                color="yellow",
                bbox=dict(facecolor="black", alpha=0.5, edgecolor="none"),
            )

    plt.tight_layout()
    if save_path:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150)
        print(f"Saved visualization to {save_path}")
    else:
        plt.show()

In [ ]:
visualize_vindr_boxes(
    "0005e8e3701dfb1dd93d53e2ff537b6e",
    "/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/train",
    "/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/annotations/annotations_train.csv",
    "/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/train_meta.csv",
)

In [ ]:
visualize_vindr_boxes(
    "0007d316f756b3fa0baea2ff514ce945",
    "/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/train",
    "/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/annotations/annotations_train.csv",
    "/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/train_meta.csv",
)

In [ ]:
img_lbl_test.columns[1:].values

In [23]:
with open("/home/harsh/Documents/fau/thesis/thesis-codebase/ALBEF/vindr_zero_shot_results_a40/vindr_zero_shot_all_checkpoints.json", "r") as f:
    zsclf = json.load(f)

In [25]:
pd.DataFrame(zsclf)

,checkpoint_09.pth,checkpoint_19.pth,checkpoint_29.pth,checkpoint_best.pth
checkpoint,output_mimic_a40_transformations/checkpoint_09...,output_mimic_a40_transformations/checkpoint_19...,output_mimic_a40_transformations/checkpoint_29...,output_mimic_a40_transformations/checkpoint_be...
num_images,3000,3000,3000,3000
label_names,"[Aortic enlargement, Atelectasis, Calcificatio...","[Aortic enlargement, Atelectasis, Calcificatio...","[Aortic enlargement, Atelectasis, Calcificatio...","[Aortic enlargement, Atelectasis, Calcificatio..."
classification,{'per_label_auc': {'Aortic enlargement': 0.680...,{'per_label_auc': {'Aortic enlargement': 0.565...,{'per_label_auc': {'Aortic enlargement': 0.682...,{'per_label_auc': {'Aortic enlargement': 0.534...
threshold,0.5,0.5,0.5,0.5


In [31]:
# -------------------------
# 1. SUMMARY TABLE
# -------------------------
summary_rows = []
ckpt_rename = {
    "checkpoint_09.pth": "ckpt 10",
    "checkpoint_19.pth": "ckpt 20",
    "checkpoint_29.pth": "ckpt 30",
    "checkpoint_best.pth": "ckpt best avg loss"
}

for ckpt_name, ckpt_data in zsclf.items():
    cls = ckpt_data["classification"]

    summary_rows.append({
        "checkpoint": ckpt_rename[ckpt_name],
        "macro_auc": cls["macro_auc"],
        "micro_auc": cls["micro_auc"],
        "map_at_10": cls["map_at_10"],
    })

df_summary = pd.DataFrame(summary_rows)
df_summary = df_summary.set_index("checkpoint")
df_summary

,macro_auc,micro_auc,map_at_10
checkpoint,,,
ckpt 10,0.791593,0.805535,0.694663
ckpt 20,0.685404,0.709694,0.699887
ckpt 30,0.764971,0.714252,0.724670
ckpt best avg loss,0.627884,0.682266,0.600726


In [32]:
df_summary.to_csv("../results/zero_shot_clf_vindr/auc_summary_24_11_2025.csv")

In [26]:
# -------------------------
# 2. PER-LABEL AUC TABLE
# -------------------------

per_label_dict = {}

for ckpt_name, ckpt_data in zsclf.items():
    per_auc = ckpt_data["classification"]["per_label_auc"]

    # Convert None → NaN for readability
    row = {label: (auc if auc is not None else float("nan"))
           for label, auc in per_auc.items()}

    per_label_dict[ckpt_name] = row

# Build DataFrame: rows = labels, columns = checkpoints
df_per_label = pd.DataFrame(per_label_dict)
df_per_label

,checkpoint_09.pth,checkpoint_19.pth,checkpoint_29.pth,checkpoint_best.pth
Aortic enlargement,0.680968,0.565571,0.682116,0.534836
Atelectasis,0.898709,0.752861,0.906123,0.479510
Calcification,0.668045,0.551500,0.647979,0.511166
Cardiomegaly,0.910069,0.833852,0.899558,0.811663
Clavicle fracture,0.882088,0.516678,0.662275,0.508339
Consolidation,0.909930,0.729102,0.893889,0.650455
Edema,NaN,NaN,NaN,NaN
Emphysema,0.761428,0.577244,0.724725,0.805472
Enlarged PA,0.844836,0.428685,0.603777,0.474098
ILD,0.849474,0.569349,0.732220,0.427717


In [29]:
df_per_label.rename({
    "checkpoint_09.pth": "ckpt 10",
    "checkpoint_19.pth": "ckpt 20",
    "checkpoint_29.pth": "ckpt 30",
    "checkpoint_best.pth": "ckpt best avg loss"
}, axis="columns", inplace=True)

In [30]:
df_per_label

,ckpt 10,ckpt 20,ckpt 30,ckpt best avg loss
Aortic enlargement,0.680968,0.565571,0.682116,0.534836
Atelectasis,0.898709,0.752861,0.906123,0.479510
Calcification,0.668045,0.551500,0.647979,0.511166
Cardiomegaly,0.910069,0.833852,0.899558,0.811663
Clavicle fracture,0.882088,0.516678,0.662275,0.508339
Consolidation,0.909930,0.729102,0.893889,0.650455
Edema,NaN,NaN,NaN,NaN
Emphysema,0.761428,0.577244,0.724725,0.805472
Enlarged PA,0.844836,0.428685,0.603777,0.474098
ILD,0.849474,0.569349,0.732220,0.427717


In [33]:
df_per_label.to_csv("../results/zero_shot_clf_vindr/per_label_auc_24_11_2025.csv")